### Git Commit SHA Integration with MLflow:

This snippet ensures every ML experiment run is traceable to the exact code version. The helper `get_git_commit_sha()` retrieves the current Git commit SHA, and if unavailable, `generate_commit_sha()` creates a synthetic SHA using commit metadata and a SHA‑1 hash. Within an MLflow run, the commit SHA is attached as a tag (`mlflow.set_tag("git_commit", commit_sha)`), linking model training directly to the source code version. This design is recruiter‑friendly because it demonstrates strong MLOps discipline—combining reproducibility, version control, and experiment tracking—showing that you can build production‑ready pipelines where models are always tied back to the code that generated them.


In [2]:
import mlflow
import subprocess
import hashlib
import time

In [ ]:


def get_git_commit_sha() -> str:
    """Return current Git commit SHA, or 'unknown' if unavailable."""
    try:
        sha = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], stderr=subprocess.STDOUT
        )
        return sha.decode("utf-8").strip()
    except Exception:
        return "unknown"

In [ ]:
def generate_commit_sha(author="Abhishek <abhishek@example.com>", message="Commit"):
    """
    Return the actual Git commit SHA if available,
    otherwise fall back to a synthetic SHA.
    """
    sha = get_git_commit_sha()
    if sha != "unknown":
        return sha

    # Fallback: synthetic SHA
    commit_content = f"""tree 4b825dc642cb6eb9a060e54bf8d69288fbee4904
author {author} {int(time.time())} +0000
committer {author} {int(time.time())} +0000

{message}
"""
    header = f"commit {len(commit_content)}\0"
    store = header + commit_content
    return hashlib.sha1(store.encode("utf-8")).hexdigest()

In [5]:
# Example MLflow usage
with mlflow.start_run():
    commit_sha = generate_commit_sha()
    mlflow.set_tag("git_commit", commit_sha)   # ✅ attach SHA as a tag
    print(f"Tagged MLflow run with commit SHA: {commit_sha}")

Tagged MLflow run with commit SHA: 9bee16273ec80a52fce75982ee807f7e8b9aa4bf
